# 74. 제품별 정상 기준을 가진 공동 가스 군집 앙상블
CN7정상695+가스10,RG3정상601+가스19를공동학습한다. 외부평가구간을빼고제품별정상흔들림기준을맞춘절대편차4개와자기노이즈비율8개두뷰를만든다. 가스2모드전문가도각학습에서새로만든다. 내부재현율우선선택. 이미본CN7/RG3개발자료의중첩검증이다.
새CN7정상1276개의정답은이공동학습에사용하지않는다. 외부구간정답은제품기준·군집·모델·임계값결정에서제외한다. CN7에는한가스이벤트만있어해당구간제외때CN7가스학습이없다는한계를표시한다.


In [ ]:
from pathlib import Path
import json,numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
s=''.join(json.loads((root/'notebooks/71_rg3_to_cn7_extra_blind_predictions.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
s=s.replace('meta=meta[meta.known.isna()]','meta=meta[meta.known.eq(True)]').replace('assert len(meta)==1276','assert len(meta)==713').replace('cn7_transfer1276_candidate_input_audit.csv','cn7_joint_input_audit.csv')
exec(s.split('memo={}')[0])
rg=train.copy();rg_ratio=ratio_train.copy()
lab=pd.read_csv(root/'data/original_split/moldset_labeled_cn7.csv',usecols=['EQUIP_CD','PART_NAME','TimeStamp','PassOrFail','Reason'])
lab=lab[lab.PART_NAME.str.contains('RH',na=False)].copy();lab.TimeStamp=pd.to_datetime(lab.TimeStamp)
cn=test.merge(lab,on=['EQUIP_CD','PART_NAME','TimeStamp'],validate='one_to_one')
cn_ratio=pd.DataFrame(ratios).replace([np.inf,-np.inf],np.nan)
select=cn.PassOrFail.eq(0)|(cn.PassOrFail.eq(1)&cn.Reason.fillna('').str.contains('가스'))
cn=cn[select].reset_index(drop=True);cn_ratio=cn_ratio[select.to_numpy()].reset_index(drop=True)
d=pd.concat([rg,cn],ignore_index=True)
ratio=pd.concat([rg_ratio,cn_ratio],ignore_index=True)
order=np.argsort(d.TimeStamp.to_numpy(),kind='stable')
d=d.iloc[order].reset_index(drop=True);ratio=ratio.iloc[order].reset_index(drop=True)
y=d.PassOrFail.to_numpy(int)
gap=d.groupby('product').TimeStamp.diff().dt.total_seconds()
boundary=d['product'].ne(d['product'].shift())|gap.gt(1800)|gap.isna()
groups=boundary.cumsum();blocks=groups.unique()
stdcols=['Barrel_Temperature_2__past10_std','Hopper_Temperature__past10_std','Cushion_Position__past10_std','Barrel_Temperature_5__past10_std']
def views(ref,q):
 normal=ref[y[ref]==0]
 a=np.empty((len(ref),4));b=np.empty((len(q),4))
 for product in d['product'].unique():
  rn=normal[d['product'].iloc[normal].eq(product).to_numpy()]
  if not len(rn):rn=normal
  im=SimpleImputer(strategy='median',keep_empty_features=True).fit(d.iloc[rn][stdcols])
  z=im.transform(d.iloc[rn][stdcols])
  center=np.median(z,axis=0);scale=np.maximum(np.quantile(z,.75,axis=0)-np.quantile(z,.25,axis=0),np.maximum(np.abs(center)*.1,1e-6))
  m=d['product'].iloc[ref].eq(product).to_numpy();n=d['product'].iloc[q].eq(product).to_numpy()
  a[m]=np.abs((im.transform(d.iloc[ref[m]][stdcols])-center)/scale) if m.any() else np.empty((0,4))
  b[n]=np.abs((im.transform(d.iloc[q[n]][stdcols])-center)/scale) if n.any() else np.empty((0,4))
 im=SimpleImputer(strategy='median',keep_empty_features=True).fit(ratio.iloc[normal])
 z=im.transform(ratio.iloc[normal]);sc=RobustScaler().fit(z)
 return [(a,b),(sc.transform(im.transform(ratio.iloc[ref])),sc.transform(im.transform(ratio.iloc[q])))]
memo={}
def predict(ref,q):
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 if len(np.unique(y[ref]))<2:return np.zeros((len(q),2))
 global_views=[];mode_views=[]
 for tr,te in views(ref,q):
  pos=np.flatnonzero(y[ref]==1);normal=np.flatnonzero(y[ref]==0)
  k=min(2,len(np.unique(tr[pos],axis=0)))
  labels=KMeans(n_clusters=k,n_init=20,random_state=42).fit_predict(tr[pos]) if k>1 else np.zeros(len(pos),int)
  glob=[];modes=[]
  for cls in [RandomForestClassifier,ExtraTreesClassifier]:
   m=cls(n_estimators=100,max_depth=5,min_samples_leaf=3,class_weight='balanced',random_state=42,n_jobs=2)
   glob.append(m.fit(tr,y[ref]).predict_proba(te)[:,1])
   sub=[]
   for label in np.unique(labels):
    ids=np.r_[normal,pos[labels==label]]
    m=cls(n_estimators=100,max_depth=5,min_samples_leaf=3,class_weight='balanced',random_state=42,n_jobs=2)
    sub.append(m.fit(tr[ids],y[ref][ids]).predict_proba(te)[:,1])
   modes.append(np.max(sub,axis=0))
  global_views.append(np.mean(glob,axis=0));mode_views.append(np.mean(modes,axis=0))
 memo[key]=np.column_stack([np.max(global_views,axis=0),np.max(mode_views,axis=0)])
 return memo[key]
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(groups.eq(b));ref=np.flatnonzero(~groups.eq(b))
 out=predict(ref,q);inner=np.full((len(ref),2),np.nan)
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy();inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for k,name in enumerate(['product_normalized_global','product_normalized_modes']):
  for budget in [.02,.05,.1]:
   cut=float(np.quantile(inner[y[ref]==0,k],1-budget,method='higher'))
   ip=inner[:,k]>cut;tn,fp,fn,tp=confusion_matrix(y[ref],ip,labels=[0,1]).ravel()
   recall=tp/(tp+fn) if tp+fn else 0;f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
   candidates.append((recall,f1,-fp,name,budget,out[:,k]>cut))
   for j,i in enumerate(q):records.append(dict(model=name,budget=budget,product=d['product'].iloc[i],TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(out[j,k]>cut)))
 selected=max(candidates,key=lambda a:a[:3])
 choices.append(dict(block=int(b),selected=selected[3],budget=selected[4],inner_recall=selected[0]))
 for j,i in enumerate(q):records.append(dict(model='inner_recall_first_selected',budget=0,product=d['product'].iloc[i],TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(selected[5][j])))
 print('Product-conditioned joint outer block',b,flush=True)
p=pd.DataFrame(records);rows=[]
for (name,budget,product),g in p.groupby(['model','budget','product']):
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 rows.append(dict(model=name,budget=budget,product=product,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),recall=tp/(tp+fn) if tp+fn else np.nan,F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices')]:
 obj.to_csv(tables/f'gas_product_normalized_joint_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))
print('Known-data nested joint development. New1276 labels excluded from this fitting. NOT fresh independent validation.')
